# SAM3 — complete the missing constraint matrix

**Refresh this notebook, then Run all.** GeoFWI3D, **fold0 / seed0**, **five training volumes**,
**25 total updates per arm**, no augmentation. Each final model is evaluated once on the same
five training-support and96 validation volumes. There are **11 new runs**:

- sdf
- bands + sdf
- bands + edge + sdf
- depth
- bands + depth
- edge + depth
- sdf + depth
- bands + edge + depth
- bands + sdf + depth
- edge + sdf + depth
- bands + edge + sdf + depth

Native SAM3 training loss is included in every arm; legacy `dice_` identifiers do not mean Dice-only loss.
Baseline, bands, original bands+edge and adapted-calibration bands+edge are reused from their saved
archives. This notebook does not repeat them or launch another edge-only tuning experiment.

### What we learned and what this experiment tests

The original bands+edge scored52.00% Dice, adapted calibration62.53%, and bands alone68.41%.
The adapted edge recipe improved a failed result, but still increased missed foreground and did not
beat bands. The purpose is to evaluate constraints fairly and preserve negative findings, not to force
one constraint to win.

- Initial gradient calibration can become inappropriate as SAM3 adapts.
- A10% median target /50% p95 calibration cap is **not a persistent gradient bound**.
- Positive gradient alignment does not establish validation benefit.
- Five diagnostic anchors are not the same distribution as the20 calibration slabs.
- Matching the recipe does not guarantee bitwise-identical GPU trajectories; CPU tests do not prove that.

### Common recipe for the missing arms

Updates1–5 use native+bands for arms containing bands, or native loss for arms without bands.
The saved bands coefficient is retained wherever bands is present. After update5, all other selected
terms are calibrated sequentially in the established order **edge → SDF → depth** on the original20
training slabs, targeting10% median with50% p95 cap. Updates6–25 use the resulting fixed coefficients
and the original global constraint ramp. No hard online cap, gradient projection or validation-driven tuning
is added. Monitoring at updates10 and25 records individual and combined auxiliary gradient strength
on the same five anchors. Compare those five anchors across stages, not the20-slab median to a5-slab median.

Existing controls used earlier calibration policies, so this is a transparent exploratory extension,
not a fully matched factorial comparison. Report all arms, including worse outcomes.

One worker runs at a time. Every completed arm is archived and its checkpoint removed before continuing.
Use the same GPU/software as the preceding runs. All new outputs go to
`results_5vol_25updates_missing11_warm5_v1`; earlier folders are preserved.


In [ ]:
from google.colab import drive, auth
drive.mount('/content/drive')
auth.authenticate_user()
from pathlib import Path
import hashlib, json, os, signal, subprocess, sys, time
NOTEBOOK_STARTED=time.monotonic()
HOME_DIR=Path('/content/drive/MyDrive/hippo_sam3_geofwi_exploration')
INPUT=HOME_DIR/'input'
SOURCE_RESULTS=HOME_DIR/'results_5vol_25updates_bands10_edge10_v1'
PREVIOUS_RESULTS=HOME_DIR/'results_5vol_24conditions_v1'
WARM_RESULTS=HOME_DIR/'results_5vol_25updates_warm5_edge10_v1'
RESULTS=HOME_DIR/'results_5vol_25updates_missing11_warm5_v1'
BUNDLE=Path('/content/hippo_sam3_geofwi_two_runs_bundle')
BASE_WEIGHTS=Path('/content/drive/MyDrive/hippo_medsam3_msd_extension/input/sam3.pt')
# Change BASE_WEIGHTS if your existing sam3.pt is elsewhere; it is never overwritten.
RESULTS.mkdir(parents=True,exist_ok=True)
print('Results and recovery:',RESULTS)


## Stage verified inputs on the local SSD
Files already present with matching hashes are reused. Different files cause a stop; active work is preserved.
The data ZIP contains only the exact five training and 96 validation volumes, not the full dataset.


In [ ]:
EXPECTED_UPLOADS={'sam3_geofwi_101vol_data.zip': '094456253ed3bf63e792d08b53eeb0c8bd36fc577a857c6cc6557d6f8271c5f0', 'sam3.pt': '9999e2341ceef5e136daa386eecb55cb414446a00ac2b55eb2dfd2f7c3cf8c9e', 'sam3_geofwi_two_runs_code.zip': '3c7ea0c2f8b95d1dff0bd14d937f7d0238f8379b81ca07dc7fb1643a2b1f4536'}
EXPECTED_MANIFEST='7a8904c0151702fced8c1a922a760efe9f2c7b80afe0f599c1f2cd34b2612684'

import fcntl, shutil, zipfile
def sha(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda:stream.read(8*1024**2),b''):h.update(block)
    return h.hexdigest()
def safe(name):
    p=Path(name)
    if p.is_absolute() or '..' in p.parts:raise ValueError('Unsafe ZIP path')
    return p
def install_zip(path,expected,allowed):
    if sha(path)!=expected:raise ValueError('ZIP hash mismatch: '+str(path))
    with zipfile.ZipFile(path) as archive:
        if len(archive.namelist())!=len(set(archive.namelist())):raise ValueError('Duplicate ZIP members')
        if set(archive.namelist())!=set(allowed):raise ValueError('ZIP inventory mismatch')
        for info in archive.infolist():
            safe(info.filename)
            if (info.external_attr>>16)&0o170000==0o120000:raise ValueError('Symlink in ZIP')
        for name,expected_sha in allowed.items():
            target=BUNDLE/safe(name)
            if target.exists():
                if target.is_symlink() or sha(target)!=expected_sha:raise ValueError('Existing file differs: '+str(target))
                continue
            target.parent.mkdir(parents=True,exist_ok=True)
            tmp=target.with_name(target.name+'.staging')
            with archive.open(name) as source,tmp.open('wb') as dest:shutil.copyfileobj(source,dest)
            if sha(tmp)!=expected_sha:raise ValueError('Extracted member differs')
            tmp.replace(target)
(BUNDLE/'work').mkdir(parents=True,exist_ok=True)
with (BUNDLE/'work/sweep.lock').open('a') as lock:
    fcntl.flock(lock,fcntl.LOCK_EX|fcntl.LOCK_NB)
    package=INPUT/'sam3_geofwi_two_runs_code.zip'
    if sha(package)!=EXPECTED_UPLOADS[package.name]:raise ValueError('Code ZIP changed')
    with zipfile.ZipFile(package) as archive:
        raw=archive.read('bundle_manifest.json')
        if hashlib.sha256(raw).hexdigest()!=EXPECTED_MANIFEST:raise ValueError('Scientific manifest differs')
        files=json.loads(raw);files['bundle_manifest.json']=EXPECTED_MANIFEST
    install_zip(package,EXPECTED_UPLOADS[package.name],files)
    data_files={'data/'+n:h for n,h in json.loads((BUNDLE/'data_manifest.json').read_text()).items()}
    if any(not (BUNDLE/n).exists() or sha(BUNDLE/n)!=h for n,h in data_files.items()):
        install_zip(INPUT/'sam3_geofwi_101vol_data.zip',EXPECTED_UPLOADS['sam3_geofwi_101vol_data.zip'],data_files)
    base=BUNDLE/'weights/sam3.pt';base.parent.mkdir(exist_ok=True)
    if not base.exists():
        if shutil.disk_usage(BUNDLE).free<16*1024**3:raise RuntimeError('Need16 GiB free runtime SSD before staging')
        if sha(BASE_WEIGHTS)!=EXPECTED_UPLOADS['sam3.pt']:raise ValueError('Existing sam3.pt differs')
        tmp=base.with_suffix('.staging');shutil.copyfile(BASE_WEIGHTS,tmp)
        if sha(tmp)!=EXPECTED_UPLOADS['sam3.pt']:raise ValueError('Checkpoint copy differs')
        tmp.replace(base)
    elif sha(base)!=EXPECTED_UPLOADS['sam3.pt']:raise ValueError('Local SAM3 checkpoint differs')
    provenance=RESULTS/'source'
    provenance.mkdir(exist_ok=True)
    dest=provenance/package.name
    if not dest.exists():shutil.copyfile(package,dest)
    if sha(dest)!=EXPECTED_UPLOADS[package.name]:raise ValueError('Durable source package differs')
print('Verified frozen source,101 volumes and SAM3 weights on local SSD.')


## Environment and commands
Reuses matching installed versions; otherwise installs the same pinned dependencies as the MSD notebook.
An A100/H100-class BF16 GPU with at least 40 GB VRAM is required. Colab speed and CUDA equivalence
are checked on your actual GPU; local packaging tests cannot establish runtime speedups.


In [ ]:
VENV=Path('/content/hippo_sam3_geofwi_venv');PYTHON=VENV/'bin/python'
def setup(args):
    p=subprocess.run(list(map(str,args)),stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
    with (RESULTS/'setup.log').open('a') as log:log.write(p.stdout)
    print('\n'.join(p.stdout.splitlines()[-30:]))
    if p.returncode:raise RuntimeError('Setup failed; see setup.log')
if not PYTHON.exists():setup([sys.executable,'-m','venv','--without-pip','--system-site-packages',VENV])
if subprocess.run([str(PYTHON),'-m','pip','--version'],capture_output=True).returncode:
    import urllib.request
    path=Path('/content/sam3_get_pip.py')
    urllib.request.urlretrieve('https://bootstrap.pypa.io/get-pip.py',path)
    setup([PYTHON,path])
probe="import importlib.metadata as m,sys; from pathlib import Path; lines=Path(sys.argv[1]).read_text().splitlines(); ok=all(m.version(s.split('==')[0])==s.split('==')[1] if '==' in s else bool(m.version(s)) for s in lines if s.strip()); sys.exit(0 if ok else 1)"
if subprocess.run([str(PYTHON),'-c',probe,str(BUNDLE/'requirements-colab.txt')],capture_output=True).returncode:
    setup([PYTHON,'-m','pip','install','--disable-pip-version-check','-r',BUNDLE/'requirements-colab.txt'])
else:print('Reusing matching dependency versions')
def execute(args,log_name,parallel=False):
    p=subprocess.Popen(list(map(str,args)),start_new_session=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    with (RESULTS/log_name).open('a') as log:
        while True:
            try:
                for line in p.stdout:print(line,end='',flush=True);log.write(line);log.flush()
                code=p.wait();break
            except KeyboardInterrupt:
                print('Clean stop requested. Keep this VM connected until the cell returns and recovery is verified.')
                if p.poll() is None:
                    if parallel:p.send_signal(signal.SIGUSR1)
                    else:os.killpg(p.pid,signal.SIGINT)
    if code:raise RuntimeError('Command failed; see '+str(RESULTS/log_name))
def campaign(command):
    execute([PYTHON,BUNDLE/'runtime.py',command,'--bundle',BUNDLE,'--durable',RESULTS],command+'.log')
campaign('verify')
subprocess.run(['nvidia-smi'],check=True)


## Stage the missing-arm runner

The existing code/data/weights bundle is reused. The new runner activates the existing SDF and depth
geometry for each applicable arm and checks that the required fields are finite and nonzero.
All four reference archives are verified before new training starts.


In [ ]:
CAMPAIGN_FILES={'sam3_missing_constraints_v1.py': 'a54aca6583556e21ec372ad1a9fb4495bbd7222a8f09c067f5d197830f8bb247', 'sam3_edge_diagnose_v1.py': '2121b3831a71fbd5b17c966a9d701e73a1a2ee066dd1fb5921b1585014bde2ae'}
for name,expected in CAMPAIGN_FILES.items():
    source=INPUT/name
    if sha(source)!=expected:raise ValueError('Campaign source differs: '+name+'; refresh the notebook')
    destination=Path('/content')/name
    shutil.copyfile(source,destination)
    if sha(destination)!=expected:raise ValueError('Campaign copy failed: '+name)
CAMPAIGN_SCRIPT=Path('/content/sam3_missing_constraints_v1.py')
print('Ready:11 missing arms, fold0 seed0,5 volumes,25 updates each; four saved references reused.')


## Run or resume the missing arms

This cell runs every unfinished arm sequentially. It prioritizes interrupted work, skips verified completed
arms, and updates the comparison after each completion. Each child process releases its GPU memory before
the next model loads. Calibration uses training support only; final evaluation is the slowest phase.

Interrupt once and wait for the clean-stop message before disconnecting. Local checkpoints are saved
after each update, with verified Drive snapshots periodically and at calibration/monitoring boundaries.
An abrupt VM loss with unsaved charged updates stops for recovery rather than silently repeating updates.
Audited evaluation feature reuse falls back to the original implementation when its checks fail.


In [ ]:
execute([PYTHON,CAMPAIGN_SCRIPT,'--bundle',BUNDLE,'--source-results',SOURCE_RESULTS,
         '--previous-results',PREVIOUS_RESULTS,'--warm-results',WARM_RESULTS,
         '--durable',RESULTS],'missing_constraints.log')


## All completed results and gradient monitoring

The table keeps saved controls and every completed new arm, regardless of ranking. Dice is the same
per-volume mean used previously; FN/FP are foreground voxel counts. Monitoring compares the identical
five anchors at updates5,10,25. The full20-slab calibration and selected coefficients remain in each
arm's `calibration.json`. Gradient norms are not shares of the Adam optimizer update.


In [ ]:
import pandas as pd
import numpy as np
path=RESULTS/'comparison.json'
if path.exists():
    result=json.loads(path.read_text())
    print(f"New arms completed: {result['new_completed']}/{result['new_total']}; references reused: {result['reused_runs']}")
    table=pd.DataFrame(result['rows']);display(table)
    table.to_csv(RESULTS/'comparison.csv',index=False)
    print(result['interpretation'])
else:print('No results yet; run the campaign cell.')
summaries=[]
for path in sorted((RESULTS/'runs').glob('*/calibration.json')):
    cal=json.loads(path.read_text())
    anchors={(r['case'],r['anchor']) for r in cal['monitoring']['10']}
    groups=[(5,[r for r in cal['rows'] if (r['case'],r['anchor']) in anchors])]
    groups += [(int(step),rows) for step,rows in cal['monitoring'].items()]
    for step,rows in sorted(groups):
        for name in cal['names'][1:]:
            full=[r['full_strength'][name]['relative_norm'] for r in rows]
            scheduled=[r['scheduled_strength'][name]['relative_norm'] for r in rows]
            summaries.append(dict(run=path.parent.name,after_update=step,term=name,slabs=len(rows),
                coefficient=cal['weights'][name],full_median=float(np.median(full)),full_max=max(full),
                scheduled_median=float(np.median(scheduled)),
                conflicting_slabs=sum(r['full_strength'][name]['cosine'] is not None and r['full_strength'][name]['cosine']<0 for r in rows),
                combined_auxiliary_over_native_median=float(np.median([r['full_strength']['combined_auxiliary_over_native'] for r in rows]))))
if summaries:
    monitoring=pd.DataFrame(summaries);display(monitoring)
    monitoring.to_csv(RESULTS/'constraint_monitoring.csv',index=False)
print('Results and recovery:',RESULTS)
